# Dataset Query & Data Loading

FanInSAR provides two query interfaces for spatial data:

| Interface | Return Type | Use Case |
|---|---|---|
| `__getitem__` (indexing) | `dict` (numpy/tensor arrays) | ML pipelines, PyTorch DataLoader |
| `xxx_query` methods | `xr.Dataset` / `xr.DataTree` | Geoscience analysis, visualization |

Both interfaces support `BoundingBox`, `Points`, and `Polygons` queries, with
optional file selection via `indexes`.

**Design rationale**: `__getitem__` follows the
[torchgeo](https://torchgeo.readthedocs.io/) convention — returning a plain
`dict` of numpy arrays makes it directly compatible with `torch.utils.data.DataLoader`.
The explicit `xxx_query` methods return labeled xarray structures with
coordinates, attributes, and CRS metadata — better suited for exploratory
analysis and interoperability with the xarray ecosystem.

When using samplers, `to_dataloader()` converts numpy arrays to CPU torch
tensors by default. Use `tensor=False` to keep numpy arrays.


> This tutorial assumes you are running inside the project's `.venv` where all
> dependencies (including `torch` and `geopandas`) are installed.
> Example: `source .venv/bin/activate`


In [1]:
import tempfile
from datetime import datetime
from pathlib import Path
from time import sleep
import numpy as np
import rasterio
from pyproj.crs import CRS
from rasterio.transform import from_bounds

from faninsar.data.datasets import RasterDataset, TimeSeriesDataset, PairDataset
from faninsar.data.query import BoundingBox, Points, Polygons, GeoQuery
from faninsar.data import samplers
from faninsar import Acquisition, Pairs


## Setup: Create Temporary Data + Datasets

This notebook generates small GeoTIFFs on the fly so every example runs
out of the box.


In [2]:
# Create temporary GeoTIFFs for a fully runnable tutorial
tmp_dir = tempfile.TemporaryDirectory()
tmp_root = Path(tmp_dir.name)

raster_dir = tmp_root / "raster"
ts_dir = tmp_root / "timeseries"
pair_dir = tmp_root / "pairs"
for d in (raster_dir, ts_dir, pair_dir):
    d.mkdir(parents=True, exist_ok=True)


def _write_tile(path: Path, bounds: tuple[float, float, float, float], value: float) -> None:
    """Write a small GeoTIFF tile for demo purposes."""
    height = width = 8
    transform = from_bounds(*bounds, width, height)
    data = np.full((height, width), value, dtype=np.float32)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=data.dtype,
        crs=CRS.from_epsg(4326),
        transform=transform,
        nodata=0.0,
    ) as dst:
        dst.write(data, 1)


bounds = (100.0, 30.0, 100.1, 30.1)

# RasterDataset files
for idx in range(3):
    _write_tile(raster_dir / f"raster_{idx:02d}.tif", bounds, value=float(idx))

# TimeSeriesDataset files (YYYYMMDD prefix)
for date, value in [("20210101", 1.0), ("20210113", 2.0), ("20210125", 3.0)]:
    _write_tile(ts_dir / f"{date}_scene.tif", bounds, value=value)

# PairDataset files (YYYYMMDD_YYYYMMDD prefix)
for primary, secondary, value in [
    ("20210101", "20210113", 1.0),
    ("20210113", "20210125", 2.0),
]:
    _write_tile(pair_dir / f"{primary}_{secondary}_pair.tif", bounds, value=value)


class DemoTimeSeriesDataset(TimeSeriesDataset):
    pattern = "*.tif"

    @classmethod
    def parse_dates(cls, paths):
        dates = [
            np.datetime64(datetime.strptime(Path(path).stem.split("_")[0], "%Y%m%d"), "ns")
            for path in paths
        ]
        return Acquisition(dates)


class DemoPairDataset(PairDataset):
    pattern = "*.tif"

    @classmethod
    def parse_pairs(cls, paths):
        pairs = []
        for path in paths:
            parts = Path(path).stem.split("_")
            primary = np.datetime64(datetime.strptime(parts[0], "%Y%m%d"), "ns")
            secondary = np.datetime64(datetime.strptime(parts[1], "%Y%m%d"), "ns")
            pairs.append((primary, secondary))
        return Pairs(pairs)


ds = RasterDataset(root_dir=raster_dir, verbose=False)
ds_ts = DemoTimeSeriesDataset(root_dir=ts_dir, verbose=False)
ds_pair = DemoPairDataset(root_dir=pair_dir, verbose=False)

print(ds.crs)
print(ds.res)
print(ds.roi)
print(ds.files)


GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
(0.01249999999999929, 0.012500000000000178)
BoundingBox(left=100.0, bottom=30.0, right=100.1, top=30.1, crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]])
                                               paths  valid  \
0  /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn...   True   
1  /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn...   True   
2  /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn...   True   

                 file_crs       

In [3]:
# Define query geometries (reused throughout this notebook)
roi = ds.roi
width = roi.right - roi.left
height = roi.top - roi.bottom

bbox = BoundingBox(
    roi.left + 0.1 * width,
    roi.bottom + 0.1 * height,
    roi.left + 0.6 * width,
    roi.bottom + 0.6 * height,
    crs=roi.crs,
)
bbox2 = BoundingBox(
    roi.left + 0.4 * width,
    roi.bottom + 0.4 * height,
    roi.left + 0.9 * width,
    roi.bottom + 0.9 * height,
    crs=roi.crs,
)
points = Points(
    [
        (roi.left + 0.2 * width, roi.bottom + 0.2 * height),
        (roi.left + 0.8 * width, roi.bottom + 0.8 * height),
    ],
    crs=roi.crs,
)


---

# Part 1: `__getitem__` — dict Interface (torchgeo-compatible)

`ds[query]` returns a flat `dict` of numpy arrays, designed for ML pipelines.

**Supported query types**: `BoundingBox`, `Points`, `Polygons`, `GeoQuery`,
or a tuple `(query, indexes)` for file selection.

## 1.1 Basic Queries

In [4]:
# BoundingBox query
result = ds[bbox]
result

Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 895.58 files/s]


{'query': BoundingBox(left=100.01, bottom=30.01, right=100.06, top=30.060000000000002, crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]),
 'data': array([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],
 
        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],
 
        [[2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.]]], dtype=float32),
 'transform': Affine(0.01249999999999929, 0.0, 100.01,
        0.0, -0.012500000000000178, 30.060000000000002),
 'crs': <Geographic 2D CRS: EPSG:4326>
 Name: WGS 84
 Axis Info [ellipsoidal]:
 - Lat[north]: Geodetic latitude (degree)
 - Lon[east]: G

In [5]:
# Points query
result = ds[points]

result

Querying points: 100%|██████████| 3/3 [00:00<00:00, 2854.56 files/s]


{'query': Points:
             x      y
 0  100.019997  30.02
 1  100.080002  30.08
 [count=2, crs='GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'],
 'data': array([[0., 0.],
        [1., 1.],
        [2., 2.]], dtype=float32),
 'crs': <Geographic 2D CRS: EPSG:4326>
 Name: WGS 84
 Axis Info [ellipsoidal]:
 - Lat[north]: Geodetic latitude (degree)
 - Lon[east]: Geodetic longitude (degree)
 Area of Use:
 - undefined
 Datum: World Geodetic System 1984
 - Ellipsoid: WGS 84
 - Prime Meridian: Greenwich,
 'nodata': np.float64(0.0),
 'paths': ['/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/tmp7tq0vhfu/raster/raster_00.tif',
  '/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/tmp7tq0vhfu/raster/raster_01.tif',
  '/var/folders/4f/zhln6dbs3

## 1.2 File Selection with `indexes`

By default, queries read **all** files. Use tuple indexing `ds[query, indexes]`
to select a subset.

In [6]:
# Tuple syntax: ds[query, indexes]
result = ds[bbox, [0, 1, 2]]
print(len(result["paths"]))  # 3
print(result["indexes"])     # array([0, 1, 2])

# Equivalent via GeoQuery
query = GeoQuery(boxes=bbox, indexes=[0, 1, 2])
result = ds[query]

Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 504.63 files/s]


3
[0 1 2]


Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 978.45 files/s]


## 1.3 Composite GeoQuery

`GeoQuery` can combine multiple query types. When multiple types are present,
the result is a **nested** dict keyed by `"points"`, `"boxes"`, `"polygons"`.

In [7]:
query = GeoQuery(points=points, boxes=bbox, indexes=[0, 1])
result = ds[query]

print(result.keys())                   # dict_keys(['points', 'boxes'])
print(result["points"]["data"].shape)  # (2, n_points)
print(result["boxes"]["data"].shape)   # (2, height, width)

Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1352.35 files/s]

dict_keys(['points', 'boxes'])
(2, 2)
(2, 4, 4)


## 1.4 `get_indexes`: Date & Pair Selection

`TimeSeriesDataset` and `PairDataset` provide `get_indexes` to convert
domain objects (dates / pairs) into integer file indexes for `__getitem__`.

In [8]:
# TimeSeriesDataset: dates -> indexes
dates = ds_ts.dates[:2]
idxs = ds_ts.get_indexes(dates=dates)

result = ds_ts[bbox, idxs]
print(result["data"].shape)  # (2, height, width)


Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1334.49 files/s]

(2, 4, 4)


In [9]:
# PairDataset: pairs -> indexes
pairs_subset = ds_pair.pairs[:5]
idxs = ds_pair.get_indexes(pairs=pairs_subset)

result = ds_pair[bbox, idxs]
print(result["data"].shape)  # (5, height, width)

Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1686.49 files/s]

(2, 4, 4)


## 1.5 Samplers + `__getitem__`

Samplers divide a dataset's spatial extent into patches. Each yields a
`BoundingBox` (or `(BoundingBox, indexes)` when `indexes` is set) that
can be fed directly into `__getitem__`.

In [10]:
# Without indexes — yields BoundingBox
sampler = samplers.RowColSampler(ds, row_num=2, col_num=2)
for bbox in sampler:
    result = ds[bbox]
    print(result["data"].shape)  # (n_files, patch_h, patch_w)

Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 1786.58 files/s]


(3, 4, 4)


Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 2063.11 files/s]


(3, 4, 4)


Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 1760.59 files/s]


(3, 4, 4)


Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 2167.60 files/s]

(3, 4, 4)


In [11]:
# With indexes — yields (BoundingBox, indexes)
idxs = ds_ts.get_indexes(dates=dates)
sampler = samplers.RowColSampler(ds_ts, row_num=2, col_num=2, indexes=idxs)
for bbox, file_idxs in sampler:
    result = ds_ts[bbox, file_idxs]
    print(result["data"].shape)  # (n_selected, patch_h, patch_w)

Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1425.42 files/s]


(2, 4, 4)


Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 605.94 files/s]


(2, 4, 4)


Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 530.66 files/s]


(2, 4, 4)


Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 678.20 files/s]

(2, 4, 4)


## 1.6 PyTorch DataLoader Integration

Since `__getitem__` returns a `dict` and samplers provide `__len__` +
`__iter__`, they plug into `torch.utils.data.DataLoader` with a thin wrapper. The DataLoader can be easily get by `to_dataloader()` method of sampler in FanInSAR.

By default, `to_dataloader()` converts numpy arrays to CPU tensors
(`tensor=True`). Use `tensor=False` to keep numpy arrays. The conversion
scope can be controlled via `tensor_scope="data"` (default) or
`tensor_scope="all"`. You can also pass `pin_memory` and `in_order`
directly to the underlying DataLoader.


In [20]:
sampler = samplers.RowColSampler(ds, row_num=4, col_num=4)
loader = sampler.to_dataloader()

for batch in loader:
    print(batch["data"].shape)  # (n_files, height, width)
    break

Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 1593.18 files/s]


torch.Size([3, 2, 2])


In [19]:
# PairDataset: select interferometric pairs
idxs = ds_pair.get_indexes(pairs=pairs_subset)
sampler = samplers.RowColSampler(ds_pair, row_num=4, col_num=4, indexes=idxs)
loader = sampler.to_dataloader(num_workers=0)

for batch in loader:
    break

batch

Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1030.41 files/s]


{'query': BoundingBox(left=100.0, bottom=30.0, right=100.025, top=30.025, crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]),
 'data': tensor([[[1., 1.],
          [1., 1.]],
 
         [[2., 2.],
          [2., 2.]]]),
 'transform': (0.01249999999999929,
  0.0,
  100.0,
  0.0,
  -0.012500000000000178,
  30.025,
  0.0,
  0.0,
  1.0),
 'crs': <Geographic 2D CRS: EPSG:4326>
 Name: WGS 84
 Axis Info [ellipsoidal]:
 - Lat[north]: Geodetic latitude (degree)
 - Lon[east]: Geodetic longitude (degree)
 Area of Use:
 - undefined
 Datum: World Geodetic System 1984
 - Ellipsoid: WGS 84
 - Prime Meridian: Greenwich,
 'nodata': np.float64(0.0),
 'paths': ['/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/tmp7tq0vhfu/pairs/20210101_20210113_pair.t

---

# Part 2: `xxx_query` — xarray Interface

The explicit query methods return **labeled** xarray structures with
coordinates (spatial, temporal, file paths), CRS metadata, and rich
attributes — ideal for interactive analysis and the xarray ecosystem
(plotting, groupby, resampling, I/O).

| Method | Return Type |
|---|---|
| `points_query(points)` | `xr.Dataset` |
| `boxes_query(bbox)` | `xr.DataTree` |
| `polygons_query(polygons)` | `xr.DataTree` |
| `query(GeoQuery)` | `xr.DataTree` |

## 2.1 `points_query` → `xr.Dataset`

Returns an `xr.Dataset` with data variable `"data"` of shape
`(file_dim, point)`.

Coordinates include `file_path`, `file_index`, `x`, `y`. The file dimension
name depends on the dataset type: `"file"` for `RasterDataset`, `"date"` for
`TimeSeriesDataset`, `"pair"` for `PairDataset`.

In [22]:
xr_result = ds.points_query(points)
xr_result

Querying points: 100%|██████████| 3/3 [00:00<00:00, 4137.75 files/s]


<xarray.Dataset> Size: 144B
Dimensions:     (file: 3, point: 2)
Coordinates:
  * file        (file) int64 24B 0 1 2
  * point       (point) int64 16B 0 1
    file_path   (file) object 24B '/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc00...
    file_index  (file) int64 24B 0 1 2
    x           (point) float64 16B 100.0 100.1
    y           (point) float64 16B 30.02 30.08
Data variables:
    data        (file, point) float32 24B nan nan 1.0 1.0 2.0 2.0
Attributes:
    crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
    nodata:      0.0
    query_json:  {"type": "Points", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WGS_198...
    query_repr:  Points(count=2, crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROI...

In [24]:
# TimeSeriesDataset: dimension is "date", with optional date filtering
xr_result = ds_ts.points_query(points, dates=dates)
xr_result

Querying points: 100%|██████████| 2/2 [00:00<00:00, 1831.97 files/s]


<xarray.Dataset> Size: 80B
Dimensions:  (date: 2, point: 2)
Coordinates:
  * date     (date) datetime64[ns] 16B 2021-01-01 2021-01-13
  * point    (point) int64 16B 0 1
    x        (point) float64 16B 100.0 100.1
    y        (point) float64 16B 30.02 30.08
Data variables:
    data     (date, point) float32 16B 1.0 1.0 2.0 2.0
Attributes:
    crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
    nodata:      0.0
    query_json:  {"type": "Points", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WGS_198...
    query_repr:  Points(count=2, crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROI...

In [ ]:
# PairDataset: dimension is "pair"
xr_result = ds_pair.points_query(points, pairs=pairs_subset)
xr_result

Querying points: 100%|██████████| 2/2 [00:00<00:00, 1633.93 files/s]


<xarray.Dataset> Size: 80B
Dimensions:  (pair: 2, point: 2)
Coordinates:
  * pair     (pair) object 16B '20210101_20210113' '20210113_20210125'
  * point    (point) int64 16B 0 1
    x        (point) float64 16B 100.0 100.1
    y        (point) float64 16B 30.02 30.08
Data variables:
    data     (pair, point) float32 16B 1.0 1.0 2.0 2.0
Attributes:
    crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
    nodata:      0.0
    query_json:  {"type": "Points", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WGS_198...
    query_repr:  Points(count=2, crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROI...

## 2.2 `boxes_query` → `xr.DataTree`

Returns an `xr.DataTree` with data variable `"data"` of shape
`(file_dim, y, x)` (or `(file_dim, band, y, x)` for multi-band files).

- **Single `BoundingBox`**: the dataset is stored at the root. Access via
  `tree["data"]` or `tree.dataset["data"]`.
- **List of `BoundingBox`**: results are stored in children
  `bbox_0`, `bbox_1`, …


In [26]:
# Single bbox
xr_result = ds.boxes_query(bbox)

xr_result

Querying bounding box: 100%|██████████| 3/3 [00:00<00:00, 1039.91 files/s]


<xarray.DataTree 'boxes'>
Group: /
    Dimensions:      (file: 3, y: 4, x: 4)
    Coordinates:
      * file         (file) int64 24B 0 1 2
      * y            (y) float64 32B 30.09 30.08 30.07 30.06
      * x            (x) float64 32B 100.1 100.1 100.1 100.1
        file_path    (file) object 24B '/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0...
        file_index   (file) int64 24B 0 1 2
        spatial_ref  int64 8B 0
    Data variables:
        data         (file, y, x) float32 192B nan nan nan nan ... 2.0 2.0 2.0 2.0
    Attributes:
        crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
        transform:   (100.05, 0.01249999999999929, 0.0, 30.1, 0.0, -0.01250000000...
        nodata:      0.0
        query_json:  {"type": "BoundingBox", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WG...
        query_repr:  BBox(100.05, 30.05, 100.1, 30.1, crs=GEOGCS["WGS 84",DATUM["...

In [ ]:
# Multiple bboxes → children named bbox_0, bbox_1, ...
xr_result = ds.boxes_query([bbox, bbox2])

xr_result["bbox_0"].ds

100%|██████████| 2/2 [00:00<00:00, 142.38it/s]


<xarray.DatasetView> Size: 336B
Dimensions:      (file: 3, y: 4, x: 4)
Coordinates:
  * file         (file) int64 24B 0 1 2
  * y            (y) float64 32B 30.08 30.07 30.06 30.05
  * x            (x) float64 32B 100.0 100.1 100.1 100.1
    file_path    (file) object 24B '/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0...
    file_index   (file) int64 24B 0 1 2
    spatial_ref  int64 8B 0
Data variables:
    data         (file, y, x) float32 192B nan nan nan nan ... 2.0 2.0 2.0 2.0
Attributes:
    crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
    transform:   (100.03999999999999, 0.01249999999999929, 0.0, 30.09, 0.0, -...
    nodata:      0.0
    query_json:  {"type": "BoundingBox", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WG...
    query_repr:  BBox(100.03999999999999, 30.04, 100.08999999999999, 30.09, c...

In [34]:
# TimeSeriesDataset: filter by dates
xr_result = ds_ts.boxes_query(bbox, dates=dates)
print(xr_result)

# PairDataset: filter by pairs
xr_result = ds_pair.boxes_query(bbox, pairs=pairs_subset)
print(xr_result)

Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1051.47 files/s]


<xarray.DataTree 'boxes'>
Group: /
    Dimensions:      (date: 2, y: 4, x: 4)
    Coordinates:
      * date         (date) datetime64[ns] 16B 2021-01-01 2021-01-13
      * y            (y) float64 32B 30.09 30.08 30.07 30.06
      * x            (x) float64 32B 100.1 100.1 100.1 100.1
        spatial_ref  int64 8B 0
    Data variables:
        data         (date, y, x) float32 128B 1.0 1.0 1.0 1.0 ... 2.0 2.0 2.0 2.0
    Attributes:
        crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
        transform:   (100.05, 0.01249999999999929, 0.0, 30.1, 0.0, -0.01250000000...
        nodata:      0.0
        query_json:  {"type": "BoundingBox", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WG...
        query_repr:  BBox(100.05, 30.05, 100.1, 30.1, crs=GEOGCS["WGS 84",DATUM["...


Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1286.99 files/s]

<xarray.DataTree 'boxes'>
Group: /
    Dimensions:      (pair: 2, y: 4, x: 4)
    Coordinates:
      * pair         (pair) object 16B '20210101_20210113' '20210113_20210125'
      * y            (y) float64 32B 30.09 30.08 30.07 30.06
      * x            (x) float64 32B 100.1 100.1 100.1 100.1
        spatial_ref  int64 8B 0
    Data variables:
        data         (pair, y, x) float32 128B 1.0 1.0 1.0 1.0 ... 2.0 2.0 2.0 2.0
    Attributes:
        crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
        transform:   (100.05, 0.01249999999999929, 0.0, 30.1, 0.0, -0.01250000000...
        nodata:      0.0
        query_json:  {"type": "BoundingBox", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WG...
        query_repr:  BBox(100.05, 30.05, 100.1, 30.1, crs=GEOGCS["WGS 84",DATUM["...


## 2.3 `polygons_query` → `xr.DataTree`

Returns an `xr.DataTree` similar to `boxes_query`, but with an
additional `mask` child containing the rasterized polygon mask.

- Single polygon: dataset stored at the root.
- Multiple polygons: children named `"0"`, `"1"`, … each with optional
  `mask` child.


In [31]:
import geopandas as gpd
from shapely.geometry import box

gdf = gpd.GeoDataFrame(
    geometry=[box(bbox.left, bbox.bottom, bbox.right, bbox.top)],
    crs=str(bbox.crs),
)
polygons = Polygons(gdf)

xr_result = ds.polygons_query(polygons)

print(xr_result)
# DataTree with root dataset (data) and optional mask child


Querying polygons: 100%|██████████| 3/3 [00:00<00:00, 343.33 files/s]

<xarray.DataTree 'polygons'>
Group: /
│   Dimensions:      (file: 3, y: 4, x: 4)
│   Coordinates:
│     * file         (file) int64 24B 0 1 2
│     * y            (y) float64 32B 30.09 30.08 30.07 30.06
│     * x            (x) float64 32B 100.1 100.1 100.1 100.1
│       file_path    (file) object 24B '/var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0...
│       file_index   (file) int64 24B 0 1 2
│       spatial_ref  int64 8B 0
│   Data variables:
│       data         (file, y, x) float32 192B nan nan nan nan ... 2.0 2.0 2.0 2.0
│   Attributes:
│       crs:         GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2...
│       transform:   (100.05, 0.01249999999999929, 0.0, 30.1, 0.0, -0.01250000000...
│       nodata:      0.0
│       query_json:  {"type": "Polygon", "crs": "GEOGCS[\"WGS 84\",DATUM[\"WGS_19...
│       query_repr:  Polygon(crs=GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 8...
└── Group: /mask
        Dimensions:  (y: 4, x: 4)
        Data variables:
            mask

## 2.4 `query` — Unified Entry Point

The `query` method accepts any spatial query type (`BoundingBox`, `Points`,
`Polygons`, or `GeoQuery`) and dispatches to the appropriate `xxx_query`
method. Always returns `xr.DataTree`.

In [32]:
# Single type — delegates to boxes_query
xr_result = ds.query(bbox)

# Composite GeoQuery
geo_query = GeoQuery(points=points, boxes=bbox)
xr_result = ds.query(geo_query)

# TimeSeriesDataset with date filtering
xr_result = ds_ts.query(bbox, dates=dates)

Querying bounding box: 100%|██████████| 2/2 [00:00<00:00, 1093.55 files/s]


---

# API Reference

## `__getitem__` Return Value

| Query Type | Return Keys |
|---|---|
| `BoundingBox` | `query`, `data`, `crs`, `nodata`, `paths`, `indexes`, `transform` |
| `Points` | `query`, `data`, `crs`, `nodata`, `paths`, `indexes` |
| `Polygons` | `query`, `data`, `crs`, `nodata`, `paths`, `indexes`, `transform`, `mask` |
| Multi-type `GeoQuery` | Nested: `{"points": {...}, "boxes": {...}, "polygons": {...}}` |
| `list[BoundingBox]` via `GeoQuery` | `list[dict]` — one dict per bbox |

Note: if input files are multi-band, `data` includes a `band` dimension
between `file_dim` and spatial dims.

## `xxx_query` Return Value

| Method | Return Type | Data Shape | Coordinates |
|---|---|---|---|
| `points_query` | `xr.Dataset` | `(file_dim, point)` (or `(file_dim, band, point)` for multi-band) | `file_path`, `file_index`, `x`, `y` |
| `boxes_query` | `xr.DataTree` | `(file_dim, y, x)` (or `(file_dim, band, y, x)`) | `file_path`, `file_index`, `y`, `x` |
| `polygons_query` | `xr.DataTree` | `(file_dim, y, x)` + mask | `file_path`, `file_index`, `y`, `x` |
| `query` | `xr.DataTree` | depends on query type | combined |

> **`file_dim`** name varies by dataset: `"file"` (`RasterDataset`),
> `"date"` (`TimeSeriesDataset`), `"pair"` (`PairDataset`).

## Sampler Types

| Sampler | Patches |
|---|---|
| `RowSampler(ds, row_num=N)` | N horizontal strips |
| `ColSampler(ds, col_num=N)` | N vertical strips |
| `RowColSampler(ds, row_num=R, col_num=C)` | R × C grid patches |
